# Youtube summary generator using Langchain

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi,TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_chroma import Chroma 
from langchain_mistralai import ChatMistralAI,MistralAIEmbeddings 
from langchain_core.prompts import PromptTemplate 

from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
# all the helper function's used

def Get_transcript(video_id = "5COUxxTRcL0",lang=['en']):
    """
    This function fetchs youtube transcripts based on given video_id and language

    video_id = str | id of particular youtube video,
    lang = list | list of language

    Return string 
    """
    myapi = YouTubeTranscriptApi()
    try:
        res = myapi.fetch(video_id=video_id,languages=lang)
        transcript = "".join(chunk.text for chunk in res)
        print(transcript)
        return transcript
    except TranscriptsDisabled:
        print("no transcript available.")
        return " "


def Split_text_to_chunks(text,chunk_size=2000,chunk_overlap=100):
    """
    This Function splits the complete text into multiple chunks.

    chunk_size = int | num of chunks 
    chunk_overlap = int | num of overlaps reqiured
    text = str | transcript

    Return Document obj
    """
    splitter = RecursiveCharacterTextSplitter(chunk_size = chunk_size,chunk_overlap=chunk_overlap)
    chunks = splitter.create_documents([text])
    return chunks


def Store_chunk_to_Chroma_db(chunks,embedding_model,dir=r"d:\AppstoneLab-AI-intern\ChromaDB"):
    """
    This function processes the chunks and converts them to embedding and stores it to ChromaDB

    **chunks** = Document | chunks of original text 
    **embedding_model** = embedding model 
    **dir** = str | directory where you want to store vectordb data

    return VectorStore
    """
    vector_store = Chroma(
        collection_name = "Transcripts",
        embedding_function= embedding_model,
        persist_directory = dir
    )

    vector_store.add_documents(chunks)

    return vector_store


def Search_query(query,retriever,chatmodel):
    """
    This function returns the result of user query from the chat model.

    **query** = str | user's query 
    **retriever** = runnable | retriever obj
    **chatmodel** = runnable | model 

    return string
    """
    prompt = PromptTemplate(
        template = """
        You are an helpful assistant,
        you only have to ANSWER from provided context, if the context is insufficient just say get a life bro.
        {context}
        QUESTION : {question}
        """,
        input_variables = ['context','question']
    )

    retrieved_docs = retriever.invoke(query) 
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    final_prompt = prompt.invoke(
        {
            "context" : context_text,
            "question" : query
        }
    )
    results = chatmodel.invoke(final_prompt)
    return results.content

def return_context_text(retrieved_docs):
    """
    This function is a helper function that concatenates all retrived documents into a single large text

    **PARAMETERS**
    retrieved_docs = Document | returned docs

    return text
    """

    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text


In [74]:
llm = ChatMistralAI(
    model = "mistral-medium-2508",
    temperature = 1
)
print("model created")
embedding_model = MistralAIEmbeddings()
print("embedding model created")

model created
embedding model created


In [ ]:
# 1. create transcript from youtube videos
transcript = Get_transcript() 

# 2. create chunks of the recived docs
chunks = Split_text_to_chunks(text=transcript,chunk_size=1000)
print("Number of chunks : ",len(chunks))

# 3. store the chunks into Vectorstore
vector_store = Store_chunk_to_Chroma_db(chunks=chunks,embedding_model=embedding_model)

# 4. Create retriever to retrive data from the vector store.
retriever = vector_store.as_retriever(search_type="similarity",search_kwargs={"k":5})

This is a neural network that I builtfrom scratch in scratch. And in thisvideo, I'm going to explain how I evengot this thing to work. And I'm going tobuild the same neural net in four levelsof difficulty from PyTorch and Numpaiall the way to C. And finally buildingthis thing in pure scratch. And spoiler,it's going to be hard. Okay. Okay. Butlet's start with something easy.>> Level one, PyTorch.>> And let me tell you, I freaking lovePyTorch. PieTorch is the most populardeep learning library. It's so fun touse and it makes my life so much easier.But before we get all excited, let meshow you what I'm doing. I'm going tobuild a neural net that recognizes zerosand ones like these. These images are28x 28, which means the network willhave 784 inputs. These will connect to ahidden layer of about 100 neurons andfinally one output neuron, which willgive us the prediction. Now, this looksfairly small and cute here, but if youvisualize it fully, it looks prettycomplex. And remember, I got to buil

In [108]:
query = input()
results = Search_query(query,retriever=retriever,chatmodel=llm)
print(results)

A **neural network** (as described in the context) is a system inspired by the human brain that learns patterns from data. Here’s how it works in simple terms:

1. **Structure**: It’s made of layers of **neurons** (small computing units).
   - **Input layer**: Takes in data (e.g., pixels of an image).
   - **Hidden layers**: Process the data through calculations (weights, biases, and activation functions).
   - **Output layer**: Produces a prediction (e.g., "0" or "1" for digit recognition).

2. **How it learns**:
   - **Weights and biases**: Adjustable numbers that tweak how inputs affect outputs.
   - **Loss function** (e.g., *binary cross-entropy*): Measures how wrong the prediction is.
   - **Optimizer** (e.g., *stochastic gradient descent*): Uses the loss to update weights/biases, improving accuracy over time.

3. **Key idea**: By stacking layers, the network learns complex patterns (e.g., recognizing handwritten digits with high accuracy, like 99% in the PyTorch example).

In the

# Complete Chain 

In [76]:
from langchain_core.output_parsers import StrOutputParser 
from langchain_core.runnables import RunnableParallel, RunnableSequence, RunnablePassthrough, RunnableLambda

In [ ]:
# creating a prompt template
prompt = PromptTemplate(
    template = """
    You are an helpful assistant,
    you only have to ANSWER from provided context, if the context is insufficient just say get a life bro.
    {context}
    QUESTION : {question}
    """,
    input_variables = ['context','question']
)

# creating a StringOutput parser
parser = StrOutputParser()

In [110]:
parallel_chain = RunnableParallel(
    {
        "context" : retriever | RunnableLambda(return_context_text),
        "question" : RunnablePassthrough()
    }
)

final_chain = parallel_chain | prompt | llm | parser 

In [111]:
query = input()
response = final_chain.invoke(query)
print(response)

get a life bro.
